# sphinxcontrib_qthelp

sphinxcontrib-qthelp 2.0.0 is Sphinx's `qthelp` builder: HTML pages plus the Qt Help project (.qhp) and collection
(.qhcp) files that Qt's `qhelpgenerator` compiles. These cells parse both XML files, check the namespace setting and
the keywords, and the registration.

In [ ]:
# Example 1: a Qt Help build writes the pages, the help project and the collection project.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'qthelp', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
assert (out / 'Demo.qhp').exists() and (out / 'Demo.qhcp').exists() and (out / 'page.html').exists()

In [ ]:
# Example 3: index entries become Qt Help keywords, and every page is listed as a file.
import xml.etree.ElementTree as ET
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'qthelp', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
root = ET.parse(out / 'Demo.qhp').getroot()
keywords = [k.attrib['name'] for k in root.iter('keyword')]
files = [f.text for f in root.iter('file')]
print(keywords, files[:5])
assert 'widgets' in keywords and 'index.html' in files and 'page.html' in files

In [ ]:
# Example 4: the builder's registration.
import sphinxcontrib.qthelp as qthelp
from sphinxcontrib.qthelp import QtHelpBuilder

print(qthelp.__version__, QtHelpBuilder.name)
assert qthelp.__version__ == "2.0.0"
assert QtHelpBuilder.name == "qthelp" and QtHelpBuilder.embedded is True